# HS10 Data Center Classification (v2 - with NAICS)

This notebook classifies HS10 commodity codes by their relevance to AI data center construction and operations using Claude AI.

**Version 2** includes NAICS codes and descriptions to provide additional context about the industry classification of each product.

In [1]:
# Set up Anthropic API key
import os

# In PowerShell: $env:ANTHROPIC_API_KEY = "your-key-here"
# In bash: export ANTHROPIC_API_KEY="your-key-here"

if 'ANTHROPIC_API_KEY' not in os.environ:
    # Set it here temporarily (DO NOT commit this to git!)
    os.environ['ANTHROPIC_API_KEY'] = 'your-api-key-here'
    print("⚠️  API key set from notebook. Remove this before committing!")
else:
    print("✅ Using API key from environment variable")

⚠️  API key set from notebook. Remove this before committing!


In [2]:
# Import required modules
import importlib
import pandas as pd
import hs10_llm_classifier_naics
importlib.reload(hs10_llm_classifier_naics)
from hs10_llm_classifier_naics import classify_single_code, classify_batch, resume_batch_classification

ModuleNotFoundError: No module named 'anthropic'

## 1️⃣ Test Single Code Classification

Run this to test the classification on a single HS10 code with NAICS context.

In [ ]:
# Test classify_single_code with a sample HS10 code + NAICS info
code = "8542310040"
description = "PROCESSORS (INCLUDING MICROPROCESSORS): GRAPHICS PROCESSING UNITS (GPUS)"
naics_code = "334413"
naics_desc = "SEMICONDUCTOR AND RELATED DEVICE MANUFACTURING"

result = classify_single_code(code, description, naics_code, naics_desc)

print(f"HS10 Code:   {code}")
print(f"HS Desc:     {description}")
print(f"NAICS:       {naics_code} - {naics_desc}")
print(f"\nRELEVANCE:   {result['relevance']}")
print(f"CONFIDENCE:  {result['confidence']}%")
print(f"CATEGORY:    {result['primary_category']}")
print(f"USE:         {result['specific_use']}")
print(f"REASONING:   {result['reasoning']}")

## 1️⃣b Test an Ambiguous Code

Test how NAICS context helps with ambiguous products like diesel engines or pumps.

In [ ]:
# Test an ambiguous code - diesel engines
code = "8408909030"
description = "COMPRESSION-IGNITION INTERNAL COMBUSTION PISTON ENGINES, NESOI, EXCEEDING 373 KW"
naics_code = "333618"
naics_desc = "OTHER ENGINE EQUIPMENT MANUFACTURING"

result = classify_single_code(code, description, naics_code, naics_desc)

print(f"HS10 Code:   {code}")
print(f"HS Desc:     {description}")
print(f"NAICS:       {naics_code} - {naics_desc}")
print(f"\nRELEVANCE:   {result['relevance']}")
print(f"CONFIDENCE:  {result['confidence']}%")
print(f"CATEGORY:    {result['primary_category']}")
print(f"USE:         {result['specific_use']}")
print(f"REASONING:   {result['reasoning']}")

## 2️⃣ Batch Process All HS10 Codes

Run this to classify all codes. Automatically saves progress every 10 items and resumes if interrupted.

**Input file:** `unique_hs10_naics_descriptions.csv`
- Columns: `HS10`, `naics`, `associated_naics`, `long_desc`, `naics_descriptions`

**Note:** This will take several hours for ~19,000 codes. Cost estimate: ~$150

In [ ]:
# BATCH PROCESSING: Classify all HS10 codes with automatic checkpointing

results_df = resume_batch_classification(
    all_codes_file='unique_hs10_naics_descriptions.csv',
    checkpoint_file='hs10_classification_progress_v2.csv',
    hs_col='HS10',
    desc_col='long_desc',
    naics_col='naics',
    naics_desc_col='naics_descriptions',
    output_file='hs10_classification_final_v2.csv',
    delay=0.5
)

print(f"\n✅ Complete! Total classified: {len(results_df):,} codes")
print(f"Results saved to: hs10_classification_final_v2.csv")

## 3️⃣ Recovery: If Something Goes Wrong

If the process crashes or gets interrupted, simply **re-run Cell 2️⃣** above. It will:
- Skip already-completed codes
- Retry any error codes
- Continue from where it stopped

### Option: Restore from Final File (if checkpoint got corrupted)

If your checkpoint file got messed up but you have a final output file, run the cell below to restore:

In [ ]:
# RESTORE: Copy final file back to checkpoint to continue
import shutil
shutil.copy('hs10_classification_final_v2.csv', 'hs10_classification_progress_v2.csv')

# Verify restoration
df_check = pd.read_csv('hs10_classification_progress_v2.csv')
print(f"✅ Checkpoint restored: {len(df_check):,} rows")
print(f"Errors to retry: {(df_check['relevance'] == 'Error').sum():,} rows")
print("\nNow re-run the batch processing cell to continue.")

## 4️⃣ Analyze Results

After classification is complete, use these cells to explore the results.

In [ ]:
# Load final results
df = pd.read_csv('hs10_classification_final_v2.csv')
print(f"Total codes: {len(df):,}")
print(f"\nRelevance distribution:")
print(df['relevance'].value_counts())

In [ ]:
# View HIGH relevance codes
high_df = df[df['relevance'] == 'High']
print(f"HIGH relevance codes: {len(high_df):,}")
print(f"\nBy category:")
print(high_df['primary_category'].value_counts())

In [ ]:
# Sample of HIGH relevance codes
high_df[['hs10_code', 'description', 'naics_code', 'primary_category', 'specific_use', 'confidence']].head(20)

In [ ]:
# View MEDIUM relevance codes
med_df = df[df['relevance'] == 'Medium']
print(f"MEDIUM relevance codes: {len(med_df):,}")
print(f"\nBy category:")
print(med_df['primary_category'].value_counts())

In [ ]:
# Confidence distribution
print("Confidence by relevance:")
print(df.groupby('relevance')['confidence'].describe())

In [ ]:
# Low confidence codes (might want to review)
low_conf = df[(df['relevance'].isin(['High', 'Medium'])) & (df['confidence'] < 70)]
print(f"High/Medium relevance codes with confidence < 70%: {len(low_conf)}")
low_conf[['hs10_code', 'description', 'relevance', 'confidence', 'reasoning']].head(10)

In [ ]:
# Export only relevant codes (High + Medium)
relevant_df = df[df['relevance'].isin(['High', 'Medium'])]
relevant_df.to_csv('hs10_ai_relevant_codes.csv', index=False)
print(f"Exported {len(relevant_df):,} relevant codes to 'hs10_ai_relevant_codes.csv'")